# SpectraShift Week 4: seed 17
Train M2, M3, and M4 sequentially on U. Attach the newest source dataset, `spectrashift-week2-frozen`, and `spectrashift-week3-pilots`. Use GPU T4 x2 with Internet disabled.


In [ ]:
from pathlib import Path
import json, os, shutil, sys, yaml

SEED = 17
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/spectrashift-week4-seed17')
WORK.mkdir(parents=True, exist_ok=True)
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift/train/week4.py').is_file()]
if not projects:
    bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(bundles) == 1, f'Expected one source bundle, found {bundles}'
    project = WORK / 'source'
    shutil.unpack_archive(str(bundles[0]), str(project))
    projects = [project]
staging_summaries = sorted(INPUT.rglob('staging_summary.json'))
partition_manifests = sorted(INPUT.rglob('partitions.parquet'))
normalizations = sorted(INPUT.rglob('normalization.json'))
freeze_summaries = sorted(INPUT.rglob('freeze_summary.json'))
week3_summaries = sorted(INPUT.rglob('week3_run_summary.json'))
assert projects, 'No Week 4 source tree found'
projects.sort(key=lambda path: (0 if 'spectrashift-source' in str(path) else 1, len(str(path))))
assert len(staging_summaries) == len(partition_manifests) == 1
assert len(normalizations) == len(freeze_summaries) == len(week3_summaries) == 1
PROJECT = projects[0]
STAGED = staging_summaries[0].parent
MANIFEST = partition_manifests[0]
NORMALIZATION = normalizations[0]
FREEZE_SUMMARY = freeze_summaries[0]
WEEK3_SUMMARY = week3_summaries[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)
print({'seed': SEED, 'project': str(PROJECT), 'work': str(WORK)})


In [ ]:
from spectrashift.train.week4 import validate_week3_approval, validate_week4_config

staging = json.loads(staging_summaries[0].read_text())
normalization = json.loads(NORMALIZATION.read_text())
freeze = json.loads(FREEZE_SUMMARY.read_text())
week3 = validate_week3_approval(WEEK3_SUMMARY)
assert staging['complete_patches'] == 50200 and staging['incomplete_patches'] == 0
assert normalization['sha256'] == '3b1d191beccde5b85fdced38c81984377e6fb3171683a6df6a9254c38f2a9a05'
assert freeze['training_approved']
assert freeze['manifest_sha256'] == '0b36a0c6c55f34a8963719af725096dde5ab1dbab72968c13639e31d1099000a'
assert week3['week4_selection']['selected']['learning_rate'] == 0.0001
runtime_configs = []
for model_id in ('m2', 'm3', 'm4'):
    filename = f'week4_{model_id}_seed{SEED}.yaml'
    config = yaml.safe_load((PROJECT / 'configs/ssl' / filename).read_text())
    config['data']['manifest_path'] = str(MANIFEST)
    config['data']['staged_root'] = str(STAGED)
    config['data']['normalization_path'] = str(NORMALIZATION)
    config['data']['freeze_summary_path'] = str(FREEZE_SUMMARY)
    config['run']['output_dir'] = str(WORK / config['run']['id'])
    config['run']['ledger_path'] = str(WORK / 'runs.jsonl')
    validate_week4_config(config)
    runtime = WORK / filename
    runtime.write_text(yaml.safe_dump(config, sort_keys=False))
    runtime_configs.append(runtime)
print([str(path) for path in runtime_configs])


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select GPU T4 x2 before running'
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
device_arch = f'sm_{major}{minor}'
supported_arches = torch.cuda.get_arch_list()
print({'gpu': gpu_name, 'device_arch': device_arch, 'pytorch_arches': supported_arches})
assert 'T4' in gpu_name, f'Select GPU T4 x2, found {gpu_name}'
assert device_arch in supported_arches, f'{device_arch} is unsupported by this PyTorch build'


In [ ]:
from spectrashift.train.week4 import run_week4_seed

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
summary = run_week4_seed(runtime_configs, WORK, WEEK3_SUMMARY, resume_roots=[INPUT])
print(json.dumps(summary, indent=2))


In [ ]:
assert summary['week4_seed_complete'] is True and summary['seed'] == 17
assert len(summary['runs']) == 3
assert all(run['stability_gate'] and run['completion_gate'] for run in summary['runs'])
assert all(run['steps'] == 18720 and run['amp_overflow_skips'] == 0 for run in summary['runs'])
print({'week4_seed_complete': True, 'seed': 17, 'summary_path': str(WORK / 'week4_seed17_summary.json')})
